# 01 — Pipeline Big Data, Dask y calidad de datos

Este notebook explica el pipeline batch/online: ingesta, validación, transformación, carga, eventos, Dask, rechazos y alertas de calidad.

In [ ]:

from pathlib import Path
import json
import os
import subprocess
import textwrap
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt


def find_project_root(start: Path | None = None) -> Path:
    """Busca la raíz del proyecto localizando docker-compose.yml."""
    start = Path.cwd() if start is None else Path(start)
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "docker-compose.yml").exists():
            return candidate
    # Si el notebook se ejecuta desde notebooks/ antes de abrir el repo correctamente.
    return start


ROOT = find_project_root()
print(f"ROOT = {ROOT}")


def read_json(path: Path, default=None):
    path = Path(path)
    if not path.exists():
        print(f"[missing] {path}")
        return default
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def read_env(path: Path) -> dict:
    env = {}
    path = Path(path)
    if not path.exists():
        return env
    for raw in path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, v = line.split("=", 1)
        env[k.strip()] = v.strip().strip('"').strip("'")
    return env


def run_cmd(cmd: str, timeout: int = 60) -> str:
    """Ejecuta un comando shell y devuelve stdout/stderr como texto."""
    print(f"$ {cmd}")
    try:
        result = subprocess.run(
            cmd,
            shell=True,
            cwd=ROOT,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
        if result.stdout:
            print(result.stdout)
        if result.stderr:
            print(result.stderr)
        print(f"exit_code={result.returncode}")
        return (result.stdout or "") + (result.stderr or "")
    except Exception as exc:
        print(f"[error] {exc}")
        return str(exc)


def show_bar(series, title: str, ylabel: str = "valor"):
    ax = pd.Series(series).plot(kind="bar")
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.set_xlabel("")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


print("=== Pipeline Big Data: estructura de código ===")

pipeline_root = ROOT / "services" / "pipeline"
paths = [
    "main.py",
    "app/orchestrator.py",
    "app/phases/ingestion.py",
    "app/phases/validation.py",
    "app/phases/transformation.py",
    "app/phases/loading.py",
    "app/phases/aggregates.py",
    "app/events.py",
    "app/logging_setup.py",
    "app/dask_runtime.py",
    "config/validation_rules.yaml",
]

rows = []
for rel in paths:
    p = pipeline_root / rel
    rows.append({
        "archivo": rel,
        "existe": p.exists(),
        "tamaño_kb": round(p.stat().st_size / 1024, 2) if p.exists() else None,
    })

display(pd.DataFrame(rows))

print("\n=== Fases del pipeline ===")
pipeline_phases = pd.DataFrame([
    {"fase": "1. Ingesta", "archivo": "ingestion.py", "función": "Lee CSV desde MinIO/S3 y usa Dask si está configurado."},
    {"fase": "2. Validación", "archivo": "validation.py + validation_rules.yaml", "función": "Detecta campos ausentes, tipos incorrectos, enums inválidos y rangos fuera de límite."},
    {"fase": "3. Transformación", "archivo": "transformation.py", "función": "Convierte filas válidas en entidades paciente e ingreso."},
    {"fase": "4. Carga", "archivo": "loading.py", "función": "Inserta datos en PostgreSQL y rechazos en MongoDB."},
    {"fase": "5. ML tabular", "archivo": "triage_client.py", "función": "Llama a ml-triage para triaje y sospecha de enfermedad."},
    {"fase": "6. Agregados", "archivo": "aggregates.py", "función": "Actualiza métricas diarias."},
    {"fase": "7. Eventos", "archivo": "events.py", "función": "Persiste system_events en MongoDB para auditoría y dashboard."},
])
display(pipeline_phases)

print("\n=== Reglas de validación declarativas ===")
rules_path = pipeline_root / "config" / "validation_rules.yaml"
if rules_path.exists():
    print(rules_path.read_text(encoding="utf-8")[:3000])
else:
    print("No se encontró validation_rules.yaml")


## 1. Fases del pipeline

```text
CSV / JSON raw
    ↓ ingestion
DataFrame
    ↓ validation
valid_df + rejects_df
    ↓ transformation
pacientes_rows + ingresos_rows
    ↓ loading
PostgreSQL + MongoDB
    ↓ ml-triage
predictions_triage + predictions_disease
    ↓ aggregates/events
system_events + aggregates_daily
```

El archivo central es:

```text
services/pipeline/app/orchestrator.py
```

In [ ]:
pipeline_files = [
    "services/pipeline/main.py",
    "services/pipeline/app/orchestrator.py",
    "services/pipeline/app/phases/ingestion.py",
    "services/pipeline/app/phases/validation.py",
    "services/pipeline/app/phases/transformation.py",
    "services/pipeline/app/phases/loading.py",
    "services/pipeline/app/phases/aggregates.py",
    "services/pipeline/app/events.py",
    "services/pipeline/app/logging_setup.py",
    "services/pipeline/app/dask_runtime.py",
    "services/pipeline/config/validation_rules.yaml",
]

pd.DataFrame([{"archivo": p, "existe": (ROOT/p).exists()} for p in pipeline_files])


## 2. Dask como procesamiento escalable

El pipeline usa Dask en la ingesta batch para leer CSVs de forma escalable.

Servicios relacionados:

- `dask-scheduler`;
- `dask-worker`;
- `pipeline`.

El dashboard de Dask está en:

```text
http://localhost:8787
```

Aunque las pruebas trabajen con volúmenes simulados, el diseño permite escalar particiones, blocksize y workers.

In [ ]:
# Comando útil para ver servicios de Dask.
print("docker compose ps dask-scheduler dask-worker")
print("docker compose logs dask-scheduler --tail=20")


## 3. Calidad de datos

La validación está declarada en YAML:

```text
services/pipeline/config/validation_rules.yaml
```

Detecta:

- campos obligatorios ausentes;
- valores fuera de rango;
- enums incorrectos;
- tipos inválidos;
- reglas cruzadas;
- duplicados si aplica.

Ejemplo real probado:

```text
patients/quality-test-invalid.csv
```

Resultado real esperado:

```json
{
  "records_in": 5,
  "valid": 1,
  "rejected": 4,
  "rejects_persisted": 4
}
```

In [ ]:
quality_summary = {
    "records_in": 5,
    "valid": 1,
    "rejected": 4,
    "rejects_persisted": 4,
}
quality_summary


In [ ]:
show_bar({"valid": quality_summary["valid"], "rejected": quality_summary["rejected"]},
         "Calidad de datos: válidos vs rechazados", "registros")


## 4. Motivos de rechazo

En la prueba de calidad aparecieron motivos como:

- `missing_field:edad`;
- `sexo:enum`;
- `intensidad_dolor:max:10`.

Estos rechazos se persisten en MongoDB en:

```text
ingestion_rejects
```

In [ ]:
reject_reasons = pd.DataFrame([
    {"motivo": "missing_field:edad", "tipo": "campo obligatorio ausente"},
    {"motivo": "sexo:enum", "tipo": "valor no permitido"},
    {"motivo": "intensidad_dolor:max:10", "tipo": "rango inválido"},
])
reject_reasons


## 5. Eventos de dominio

El pipeline guarda eventos en MongoDB en la colección:

```text
system_events
```

Eventos importantes:

- `pipeline.run.start`;
- `pipeline.file.read`;
- `pipeline.validation.done`;
- `pipeline.transformation.done`;
- `pipeline.loading.done`;
- `pipeline.rejects.persisted`;
- `pipeline.ml.done`;
- `pipeline.run.end`.

Estos eventos alimentan el dashboard, la auditoría y el servicio `automation`.

In [ ]:
events = pd.DataFrame([
    {"event": "pipeline.run.start", "level": "info", "descripcion": "inicio de ejecución"},
    {"event": "pipeline.file.read", "level": "info", "descripcion": "CSV leído"},
    {"event": "pipeline.validation.done", "level": "warning/info", "descripcion": "resultado validación"},
    {"event": "pipeline.rejects.persisted", "level": "warning", "descripcion": "rechazos persistidos"},
    {"event": "pipeline.ml.done", "level": "warning/info", "descripcion": "triaje completado o pendiente"},
    {"event": "pipeline.run.end", "level": "info", "descripcion": "fin de ejecución"},
])
events


## 6. Comandos de demostración

Generar CSV sintético:

```powershell
docker compose exec pipeline python main.py seed --n 100 --seed 123
```

Ejecutar batch:

```powershell
docker compose exec pipeline python main.py batch --key "patients/<archivo>.csv"
```

Consultar eventos recientes:

```powershell
docker compose exec mongodb mongosh -u admin -p change-me --authenticationDatabase admin hospital --eval "db.system_events.find().sort({timestamp:-1}).limit(10).pretty()"
```